# 🛣️ SVC (máquina de vectores de soporte) — versión mínima

**Idea:** SVC busca la línea que separa a los que aprueban de los que suspenden dejando **la "calle" más ancha posible** entre los dos grupos.

Ejecuta cada celda con **Shift + Enter**, en orden.

## 1. Crear los datos
Esta celda fabrica `alumnos.csv` con 1.000 alumnos. **No hace falta entenderla.**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(12)
n = 1000
horas_estudio = np.clip(rng.gamma(4, 2.2, n), 0, 20).round(1)
asistencia = np.clip(rng.normal(80, 14, n), 40, 100).round(0)
nota_previa = np.clip(rng.normal(6, 1.8, n), 0, 10).round(1)
tareas = np.clip(np.round(asistencia / 12 + rng.normal(0, 1.5, n)), 0, 10)
z = (-7.2 + 0.18 * horas_estudio + 0.035 * asistencia + 0.35 * nota_previa + 0.12 * tareas
     + 2.5 * ((horas_estudio > 10) & (asistencia > 80))
     - 2.0 * (asistencia < 60)) * 2.0
aprueba = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

pd.DataFrame({"horas_estudio": horas_estudio, "asistencia": asistencia,
              "nota_previa": nota_previa, "tareas": tareas, "aprueba": aprueba}).to_csv("alumnos.csv", index=False)
print("✅ alumnos.csv creado")

## 2. Ver los datos
Cada fila es un alumno. `aprueba` vale 1 (aprueba) o 0 (suspende): es lo que queremos predecir.

In [ ]:
df = pd.read_csv("alumnos.csv")
df.head()

## 3. Separar entrenamiento (80 %) y examen (20 %)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["horas_estudio", "asistencia", "nota_previa", "tareas"]]
y = df["aprueba"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Escalar
SVC se basa en **distancias**, así que todas las columnas deben estar en la misma escala.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X_train = escalador.fit_transform(X_train)
X_test = escalador.transform(X_test)

## 5. Entrenar y evaluar

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

modelo = SVC()
modelo.fit(X_train, y_train)

prediccion = modelo.predict(X_test)
print("Aciertos en el examen:", accuracy_score(y_test, prediccion))

## 6. El parámetro C: ¿cuántos errores tolera?

In [ ]:
for C in [0.01, 1, 100]:
    m = SVC(C=C).fit(X_train, y_train)
    print("C =", C, "→ aciertos:", round(accuracy_score(y_test, m.predict(X_test)), 2))

👀 **C pequeño** = tolera muchos errores (modelo demasiado simple). **C grande** = casi no tolera errores (riesgo de memorizar).

## 7. Predecir un alumno nuevo

In [ ]:
nuevo = pd.DataFrame({"horas_estudio": [12], "asistencia": [90], "nota_previa": [7.5], "tareas": [9]})

resultado = modelo.predict(escalador.transform(nuevo))
print("Aprueba" if resultado[0] == 1 else "Suspende")

## Resumen
1. **SVC = buscar la frontera con la calle más ancha.**
2. **Escalar es imprescindible.**
3. **C** decide cuántos errores tolera.

## ✍️ Ejercicios
1. Cambia los datos del alumno nuevo. ¿Cuándo pasa a "Suspende"?
2. Prueba `SVC(kernel="linear")` en el paso 5 (frontera recta). ¿Mejora o empeora?